# Day 1 - Document Ingestion (AsthmaDaily RAG)
### GINA 2026 + WHO 2026 - Multi-Source Clinical RAG

This notebook follows the same steps as the original hackathon Day 1 notebook, applied to our own project (asthma): parsing two real clinical guideline PDFs, preprocessing the extracted text, comparing chunking strategies, generating embeddings, and building a queryable vector index - with the output of every step saved to `outputs/`.

**By the end of this notebook you will be able to:**
1. Explain why grounding - not raw model memory - matters in clinical AI
2. Parse two PDFs (GINA 2026, WHO 2026) and inspect their extracted structure
3. See why raw PDF text needs preprocessing before it's usable
4. Compare fixed-size vs. section-aware chunking (with overlap) on the same documents
5. Generate an embedding and explain what the resulting vector represents
6. Build a persisted, source-aware vector index and run a real query against it

> **Data source:** `data/GINA_2026.pdf` (298 pages, primary reference, all ages) and `data/WHO_2026_Asthma_Children_Adolescents.pdf` (107 pages, supporting reference, ages 0-19) - both real guideline documents, registered in `config.SOURCE_REGISTRY`.


## 0. Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import config
from pathlib import Path

print("Data directory:", config.DATA_DIR)
print("Chunk size (tokens):", config.CHUNK_SIZE)
print("Chunk overlap (tokens):", config.CHUNK_OVERLAP)
print("PDFs found:", [p.name for p in config.DATA_DIR.glob("*.pdf")])
print("Registered sources:", list(config.SOURCE_REGISTRY.keys()))


## 1. Why Grounding Matters

A large language model can generate a fluent, confident-sounding clinical recommendation **even when it has no real evidence behind it.** It has no built-in mechanism to say "I don't know."

Retrieval-Augmented Generation (RAG) fixes this by separating two things:

- **What the model knows** (its training data - broad, but unverifiable and possibly stale)
- **What the model is allowed to say** (only what's in the text we hand it right now)

Everything in this notebook is the **first half** of that separation: turning two trustworthy asthma guideline PDFs into a searchable, citable, source-aware index.


## 2. Step 1 - Load the PDFs (multi-source, with metadata)

`ingest.load_pdfs()` loads every PDF in `data/`, looks each one up in `config.SOURCE_REGISTRY`, and stamps `document_name`, `source`, `year`, `role`, `age_group` and a 1-indexed `page_number` onto every page's metadata **before** chunking, so that metadata survives all the way to the final citation. It also saves the raw pages to `outputs/01_pages_raw.json`.


In [ ]:
from ingest import load_pdfs

pages = load_pdfs()

print(f"\nTotal pages loaded: {len(pages)}")
print("\n--- Sample metadata (GINA, page 43) ---")
sample = next(p for p in pages if p.metadata['document_name']=='GINA 2026' and p.metadata['page_number']==43)
print(sample.metadata)
print("\n--- Raw text (first 400 chars) ---")
print(sample.page_content[:400])


### Checkpoint 1

Look at the raw text above.

- Are section headings visible as recognizable text?
- Do you see repeated boilerplate (running headers, page numbers, copyright notices)?

You should see some noise (e.g. `COPYRIGHTED MATERIAL - DO NOT COPY OR DISTRIBUTE`, WHO's sidebar table-of-contents text repeating on every page). That noise is exactly what Step 2 removes - this is why we don't chunk the raw text directly.


## 3. Step 2 - Preprocessing

`preprocess.preprocess_pages()` runs three real fixes before anything gets chunked:

1. **Hyphenation repair** - joins words broken across a line-wrap (`bronchodila-\ntor` -> `bronchodilator`)
2. **Unicode / whitespace normalization** - curly quotes, non-breaking spaces, collapsed runs of spaces, while keeping paragraph breaks intact
3. **Boilerplate removal** - detects lines that repeat across many pages of the *same* document (running headers/footers, sidebar nav text, bare page numbers, copyright notices) and strips them out, per source document

The cleaned pages are saved to `outputs/02_pages_preprocessed.json`.


In [ ]:
from ingest import preprocess

pages = preprocess(pages)

print("--- Same GINA page 43, AFTER preprocessing ---")
sample = next(p for p in pages if p.metadata['document_name']=='GINA 2026' and p.metadata['page_number']==43)
print(sample.page_content[:400])


### Checkpoint 2

Compare this cleaned text to the raw text in Checkpoint 1.

- Is the copyright boilerplate gone?
- Are hyphenated words rejoined?

If parsing + preprocessing look clean here, chunking (Step 3) will work well. If it still looks messy for a particular document, the fix belongs here, upstream of chunking - not in the chunker.


## 4. Step 3 - Compare Chunking Strategies (with overlap)

We build **two** chunkers on the same preprocessed pages and compare them directly: a naive fixed-size splitter with **no overlap**, and the section-aware splitter with **overlap** actually used in `ingest.py` (`chunk_size=400 tokens`, `chunk_overlap=50 tokens`).


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- Naive fixed-size splitter: no overlap, no regard for sentence/paragraph boundaries ---
naive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=0,
    separators=[""],
)
naive_chunks = naive_splitter.split_documents(pages)

# --- Section-aware splitter WITH OVERLAP: same one used in ingest.py ---
aware_splitter = RecursiveCharacterTextSplitter(
    chunk_size=config.CHUNK_SIZE * config.CHARS_PER_TOKEN,
    chunk_overlap=config.CHUNK_OVERLAP * config.CHARS_PER_TOKEN,
    separators=["\n\n", "\n", ". ", " ", ""],
)
aware_chunks = aware_splitter.split_documents(pages)

print(f"Naive fixed-size chunker (no overlap):     {len(naive_chunks)} chunks")
print(f"Section-aware chunker (with overlap):      {len(aware_chunks)} chunks")


In [ ]:
print("--- Naive chunk #200 (often cuts mid-sentence, no overlap with #201) ---")
print(repr(naive_chunks[200].page_content))

print("\n--- Section-aware chunk #200 (respects boundaries, overlaps with #201) ---")
print(repr(aware_chunks[200].page_content[:300]))
print("\n--- Start of #201, showing the overlap with the end of #200 ---")
print(repr(aware_chunks[201].page_content[:150]))


### Checkpoint 3

- Does the naive chunk end mid-word or mid-sentence?
- Can you see the tail of chunk #200 reappear at the start of chunk #201 (the overlap)?

**Why overlap matters for a clinical RAG system:** a single sharp cut can separate a recommendation from the condition it depends on (e.g. a dosage from the age group it applies to). A 50-token overlap means that boundary content still appears fully inside at least one chunk, instead of being split in half and lost.


## 5. Step 4 - Attach Citation Metadata + Topics

`chunk_documents()` builds the final chunks (section-aware, with overlap) and attaches a stable `chunk_id` and a lightweight keyword-based `topics` tag to each one - so retrieval can later filter by topic (e.g. `exacerbation`, `medication`) as well as by source. Output is saved to `outputs/03_chunks.json`.


In [ ]:
from ingest import chunk_documents

chunks = chunk_documents(pages)
print(f"Total chunks with metadata attached: {len(chunks)}\n")

sample = chunks[50]
print("--- Sample chunk metadata ---")
for k in ["document_name", "source", "age_group", "page_number", "chunk_id", "topics"]:
    print(f"  {k}: {sample.metadata.get(k)}")
print("\n--- Sample chunk text ---")
print(sample.page_content[:300])


## 6. Step 5 - What Is an Embedding, Really?

An embedding model converts text into a vector of numbers that captures meaning - texts about similar topics end up as vectors that point in similar directions, even without sharing the same words. Below we use the project's **active embedding backend** (`config.DEFAULT_EMBEDDING_MODEL_ID`) and check that a same-meaning asthma phrase is closer to another asthma phrase than to an unrelated medical topic.


In [ ]:
import numpy as np
from ingest import get_embedding_function

corpus_texts = [c.page_content for c in chunks]
embed_fn = get_embedding_function(corpus_texts=corpus_texts)

texts = [
    "inhaled corticosteroid treatment for asthma",
    "ICS therapy for airway inflammation in asthma",   # same meaning, different words
    "recommended screening interval for breast cancer", # unrelated topic
]

vectors = np.array(embed_fn.embed_documents(texts))
print(f"Each embedding is a vector of length {vectors.shape[1]}\n")

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim_related = cosine_similarity(vectors[0], vectors[1])
sim_unrelated = cosine_similarity(vectors[0], vectors[2])

print(f"Similarity - same meaning, different words:  {sim_related:.3f}")
print(f"Similarity - genuinely different topics:      {sim_unrelated:.3f}")


### Checkpoint 4

The first similarity score should be noticeably **higher** than the second. See `README.md` -> "Embedding model comparison" for how this project's offline TF-IDF+LSA baseline compares against real sentence-transformer models on the actual retrieval task (Day 2 covers that comparison in full).


## 7. Step 6 - Build the Source-Aware Vector Index

Now we embed every chunk and store it in a persisted ChromaDB collection, using `build_index()` from `ingest.py`. A summary is saved to `outputs/04_index_summary_<model_id>.json`.


In [ ]:
from ingest import build_index

vectordb = build_index(chunks)
print("\nIndex build complete.")


## 8. Step 7 - Run Real Queries (source-aware)

Two real questions: a general one (GINA is the primary source for all ages) and a children-specific one, where `source_aware_retrieve()` pulls from **both** GINA and WHO and keeps the results separated by source.


In [ ]:
from query import retrieve, source_aware_retrieve

print("=== General question - GINA only (primary source, all ages) ===")
q1 = "What is the stepwise pharmacological approach to asthma treatment?"
for doc, score in retrieve(vectordb, q1, k=3, source="GINA"):
    print(f"[{score:.3f}] {doc.metadata['document_name']} p.{doc.metadata['page_number']} "
          f"(topics={doc.metadata['topics']})")
    print(f"   {doc.page_content[:150].strip()}...\n")

print("=== Children-specific question - GINA + WHO fused ===")
q2 = "What inhaled corticosteroid doses are recommended for children with asthma?"
fused = source_aware_retrieve(vectordb, q2, age_group="0-19", k=2)
for src, results in fused.items():
    print(f"-- {src} --")
    for doc, score in results:
        print(f"   [{score:.3f}] p.{doc.metadata['page_number']}: "
              f"{doc.page_content[:120].strip()}...")


### Checkpoint 5 - Day 1 Self-Check

- [ ] Every result shows a real document name and page number (not `None`)
- [ ] You can explain, in one sentence, why we preprocess before chunking
- [ ] You can explain, in one sentence, why chunk overlap matters for this project
- [ ] You know which embedding backend is currently active (`config.DEFAULT_EMBEDDING_MODEL_ID`) and where its outputs are saved

## What's Next

Day 2's notebook picks up exactly here: tuning `top_k`, running the same chunk-size ablation on this project's own data, and - the main addition for this project - benchmarking **three different embedding models** against each other with real, logged Precision@k numbers, and writing the final comparison report.
